In [5]:
import sys
import numpy as np
from sklearn.metrics import r2_score
from scipy.stats import pearsonr

sys.path.append("../../../../")

import sherlock as slk
from sherlock.benchmarks.models.scgen._data_preprocessing import preprocess_for_scgen
from sherlock.benchmarks.models.scgen._trainer import train_scgen
from sherlock.benchmarks.models.scgen._predict import predict_scgen_manual

In [6]:
data = slk.datasets.replogle()
print(data)
print("obs cols:", list(data.obs.columns))

AnnData object with n_obs × n_vars = 118641 × 1185
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'treatment', 'pert'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    uns: 'pathways'
obs cols: ['gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'treatment', 'pert']


In [7]:
pert_col = slk.configs.get_config("pert_key")
control_label = slk.configs.get_config("ntc_label")

data_scgen = preprocess_for_scgen(
    data,
    pert_col=pert_col,
    control_label=control_label,
    min_cells=30,
    target_sum=1e4,
    inplace=False,
)

print(data_scgen)
print(data_scgen.obs[pert_col].value_counts().head(10))

AnnData object with n_obs × n_vars = 118393 × 1185
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'treatment', 'pert'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    uns: 'pathways', 'log1p'
pert
NTC           2000
PSMG3          791
CCNK           746
GPS1           671
FBXL14         648
RBM14-RBM4     630
FOLR3          610
NCBP2          547
PPP1CA         546
BOP1           492
Name: count, dtype: int64


In [8]:
test_pert = "PSMG3"
train_frac = 0.8
seed = 0
rng = np.random.default_rng(seed)

pert_mask = data_scgen.obs[pert_col].astype(str).values == test_pert
ntc_mask = data_scgen.obs[pert_col].astype(str).values == control_label

pert_idx = np.where(pert_mask)[0]
ntc_idx = np.where(ntc_mask)[0]

rng.shuffle(pert_idx)
rng.shuffle(ntc_idx)

n_pert_train = int(train_frac * len(pert_idx))
n_ntc_train = int(train_frac * len(ntc_idx))

pert_train_idx = pert_idx[:n_pert_train]
pert_test_idx = pert_idx[n_pert_train:]

ntc_train_idx = ntc_idx[:n_ntc_train]
ntc_test_idx = ntc_idx[n_ntc_train:]

other_mask = ~(pert_mask | ntc_mask)

train_mask = np.zeros(data_scgen.n_obs, dtype=bool)
train_mask[other_mask] = True
train_mask[pert_train_idx] = True
train_mask[ntc_train_idx] = True

train_data = data_scgen[train_mask].copy()
real_test = data_scgen[pert_test_idx].copy()

print("train_data:", train_data)
print("real_test:", real_test)
print("n train pert:", len(pert_train_idx))
print("n test pert:", len(pert_test_idx))
print("n train ntc:", len(ntc_train_idx))
print("n test ntc:", len(ntc_test_idx))

train_data: AnnData object with n_obs × n_vars = 117834 × 1185
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'treatment', 'pert'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    uns: 'pathways', 'log1p'
real_test: AnnData object with n_obs × n_vars = 159 × 1185
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'treatment', 'pert'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
    uns: 'pathways', 'log1p'
n train pert: 632
n test pert: 159
n train ntc: 1600
n test ntc: 400


In [9]:
model = train_scgen(
    train_data,
    pert_col=pert_col,
    control_label=control_label,
    n_epochs=20,   # raise later
)

print("trained")

/Users/mahsa/azizi_lab/PerturbVAE/.venv/lib/python3.11/site-packages/scvi/train/_trainrunner.py:98: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/mahsa/azizi_lab/PerturbVAE/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/Users/mahsa/azizi_lab/PerturbVAE/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/

Training:   0%|          | 0/20 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


trained


In [14]:
pred_adata, delta = predict_scgen_manual(
    model,
    ctrl_key=control_label,
    stim_key=test_pert,
    condition_key="condition",
    cell_type_key="cell_type",
    celltype_to_predict="all_cells",
)

print(pred_adata)
print(pred_adata.X[:5, :5])
print("delta first 10:", delta[:10])

AnnData object with n_obs × n_vars = 1600 × 1185
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'treatment', 'pert', 'condition', 'cell_type', '_scvi_batch', '_scvi_labels'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'
[[1.21744    0.30520183 2.1868954  0.75302786 3.4386063 ]
 [0.9657019  0.20886491 2.2688363  0.75761586 3.2849643 ]
 [1.3384963  0.2888697  2.313303   0.83962345 3.3793068 ]
 [0.88766664 0.26726517 2.018304   0.60225564 3.4793901 ]
 [1.0097923  0.31384653 2.2442207  0.72565055 3.5143824 ]]
delta first 10: [-0.94510823  0.33739567  0.96724737  0.07386592  0.40755293 -1.5039957
 -0.4877251   0.05934419  2.3924685  -1.2302177 ]


In [15]:
pred_mean = np.asarray(pred_adata.X.mean(axis=0)).ravel()
real_mean = np.asarray(real_test.X.mean(axis=0)).ravel()

r2 = r2_score(real_mean, pred_mean)
pearson = pearsonr(real_mean, pred_mean)[0]

print("R2:", r2)
print("Pearson:", pearson)

R2: 0.9948757886886597
Pearson: 0.9974817


In [16]:
def run_single_holdout(
    data_scgen,
    test_pert,
    pert_col="pert",
    control_label="NTC",
    train_frac=0.8,
    seed=0,
    n_epochs=20,
):
    rng = np.random.default_rng(seed)

    pert_vals = data_scgen.obs[pert_col].astype(str).values
    pert_mask = pert_vals == test_pert
    ntc_mask = pert_vals == control_label

    pert_idx = np.where(pert_mask)[0]
    ntc_idx = np.where(ntc_mask)[0]

    if len(pert_idx) < 2 or len(ntc_idx) < 2:
        raise ValueError(f"Not enough cells for {test_pert}")

    rng.shuffle(pert_idx)
    rng.shuffle(ntc_idx)

    n_pert_train = int(train_frac * len(pert_idx))
    n_ntc_train = int(train_frac * len(ntc_idx))

    n_pert_train = max(1, min(n_pert_train, len(pert_idx) - 1))
    n_ntc_train = max(1, min(n_ntc_train, len(ntc_idx) - 1))

    pert_train_idx = pert_idx[:n_pert_train]
    pert_test_idx = pert_idx[n_pert_train:]
    ntc_train_idx = ntc_idx[:n_ntc_train]

    other_mask = ~(pert_mask | ntc_mask)

    train_mask = np.zeros(data_scgen.n_obs, dtype=bool)
    train_mask[other_mask] = True
    train_mask[pert_train_idx] = True
    train_mask[ntc_train_idx] = True

    train_data = data_scgen[train_mask].copy()
    real_test = data_scgen[pert_test_idx].copy()

    model = train_scgen(
        train_data,
        pert_col=pert_col,
        control_label=control_label,
        n_epochs=n_epochs,
    )

    pred_adata, delta = predict_scgen_manual(
        model,
        ctrl_key=control_label,
        stim_key=test_pert,
        condition_key="condition",
        cell_type_key="cell_type",
        celltype_to_predict="all_cells",
    )

    pred_mean = np.asarray(pred_adata.X.mean(axis=0)).ravel()
    real_mean = np.asarray(real_test.X.mean(axis=0)).ravel()

    return {
        "perturbation": test_pert,
        "r2": float(r2_score(real_mean, pred_mean)),
        "pearson": float(pearsonr(real_mean, pred_mean)[0]),
        "n_train_pert": int(len(pert_train_idx)),
        "n_test_pert": int(len(pert_test_idx)),
        "n_train_ntc": int(len(ntc_train_idx)),
    }